# Data Cleaning Pipeline - Market Mix Modeling

This notebook implements a comprehensive data cleaning pipeline for the MMM dataset with:
1. Missing value treatment
2. Duplicate removal
3. Datatype conversion
4. Outlier detection
5. Data validation checks
6. Save cleaned dataset

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 1. Load Raw Data

In [2]:
# Load the raw dataset
data_path = '../data/raw/synthetic_mmm_weekly_india.csv'
df_raw = pd.read_csv(data_path)

# Create working copy
df = df_raw.copy()

print(f"✓ Raw dataset loaded: {data_path}")
print(f"\nInitial Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColumn Names: {list(df.columns)}")

✓ Raw dataset loaded: ../data/raw/synthetic_mmm_weekly_india.csv

Initial Dataset Shape: 11,232 rows × 28 columns
Memory Usage: 4.86 MB

Column Names: ['Week', 'Geo', 'Brand', 'SKU', 'Sales_Units', 'Sales_Value', 'MRP', 'Net_Price', 'Feature_Flag', 'Display_Flag', 'TPR_Flag', 'Trade_Spend', 'TV_Impressions', 'YouTube_Impressions', 'Facebook_Impressions', 'Instagram_Impressions', 'Print_Readership', 'Radio_Listenership', 'FB_Banner_Content_Score', 'IG_Banner_Content_Score', 'Weighted_Distribution', 'Numeric_Distribution', 'TDP', 'NOS', 'CPI', 'GDP_Growth', 'Festival_Index', 'Rainfall_Index']


## 2. Missing Value Treatment

In [3]:
print("="*80)
print("STEP 1: MISSING VALUE ANALYSIS & TREATMENT")
print("="*80)

# Check missing values
missing_before = df.isnull().sum()
missing_pct = (missing_before / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_before.values,
    'Missing_Percentage': missing_pct.values,
    'Data_Type': df.dtypes.values
})

missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_summary) > 0:
    print("\nColumns with Missing Values:")
    print(missing_summary.to_string(index=False))
else:
    print("\n✓ No missing values detected in the dataset!")

total_missing_before = missing_before.sum()
print(f"\nTotal Missing Values: {total_missing_before}")
print(f"Data Completeness: {100 - (total_missing_before / (df.shape[0] * df.shape[1]) * 100):.2f}%")

# Treatment Strategy
print("\n" + "-"*80)
print("Treatment Strategy:")
print("-"*80)

# For categorical columns: fill with mode or 'Unknown'
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
        df[col].fillna(mode_val, inplace=True)
        print(f"  → {col}: Filled with mode: '{mode_val}'")

# For numerical columns: fill with median
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols[:10]}... (showing first 10)")

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  → {col}: Filled with median: {median_val:.2f}")

total_missing_after = df.isnull().sum().sum()
print(f"\n✓ Missing Value Treatment Complete")
print(f"  Before: {total_missing_before} missing values")
print(f"  After: {total_missing_after} missing values")

STEP 1: MISSING VALUE ANALYSIS & TREATMENT

✓ No missing values detected in the dataset!

Total Missing Values: 0
Data Completeness: 100.00%

--------------------------------------------------------------------------------
Treatment Strategy:
--------------------------------------------------------------------------------

Categorical columns (4): ['Week', 'Geo', 'Brand', 'SKU']

Numeric columns (24): ['Sales_Units', 'Sales_Value', 'MRP', 'Net_Price', 'Feature_Flag', 'Display_Flag', 'TPR_Flag', 'Trade_Spend', 'TV_Impressions', 'YouTube_Impressions']... (showing first 10)

✓ Missing Value Treatment Complete
  Before: 0 missing values
  After: 0 missing values


## 3. Duplicate Removal

In [4]:
print("="*80)
print("STEP 2: DUPLICATE REMOVAL")
print("="*80)

# Check for complete duplicates
complete_duplicates = df.duplicated().sum()
print(f"\nComplete Duplicates (all columns): {complete_duplicates}")
print(f"Percentage: {complete_duplicates / len(df) * 100:.2f}%")

# Check for duplicates on key columns
if 'Week' in df.columns and 'Geo' in df.columns and 'Brand' in df.columns and 'SKU' in df.columns:
    key_cols = ['Week', 'Geo', 'Brand', 'SKU']
    key_duplicates = df.duplicated(subset=key_cols).sum()
    print(f"\nDuplicates on Key Columns {key_cols}: {key_duplicates}")
    print(f"Percentage: {key_duplicates / len(df) * 100:.2f}%")

# Remove duplicates
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"\n✓ Duplicate Removal Complete")
print(f"  Rows before: {rows_before:,}")
print(f"  Rows after: {rows_after:,}")
print(f"  Rows removed: {rows_removed:,}")

# Verify no duplicates remain
remaining_duplicates = df.duplicated().sum()
print(f"\n  Remaining duplicates: {remaining_duplicates}")

STEP 2: DUPLICATE REMOVAL

Complete Duplicates (all columns): 0
Percentage: 0.00%

Duplicates on Key Columns ['Week', 'Geo', 'Brand', 'SKU']: 0
Percentage: 0.00%

✓ Duplicate Removal Complete
  Rows before: 11,232
  Rows after: 11,232
  Rows removed: 0

  Remaining duplicates: 0


## 4. Datatype Conversion

In [5]:
print("="*80)
print("STEP 3: DATATYPE CONVERSION")
print("="*80)

print("\nCurrent Data Types:")
print(df.dtypes)

# Define optimal datatypes
print("\n" + "-"*80)
print("Optimizing Data Types for Memory Efficiency:")
print("-"*80)

# Convert object columns to category where appropriate
categorical_candidates = ['Week', 'Geo', 'Brand', 'SKU']
for col in categorical_candidates:
    if col in df.columns and df[col].dtype == 'object':
        unique_count = df[col].nunique()
        if unique_count < len(df) * 0.05:  # If less than 5% unique values
            df[col] = df[col].astype('category')
            print(f"  → {col}: object → category (Unique values: {unique_count})")

# Convert integer columns to int32 if possible (memory optimization)
int_cols = df.select_dtypes(include=['int64']).columns.tolist()
for col in int_cols:
    if df[col].min() >= 0 and df[col].max() < 2**31:  # Can fit in int32
        df[col] = df[col].astype('int32')
        print(f"  → {col}: int64 → int32")

# Convert float64 to float32 for non-critical columns (memory optimization)
float_cols = df.select_dtypes(include=['float64']).columns.tolist()
for col in float_cols:
    df[col] = df[col].astype('float32')
    print(f"  → {col}: float64 → float32")

# Flag columns conversion (0/1 to bool for efficiency)
flag_cols = [col for col in df.columns if 'Flag' in col]
for col in flag_cols:
    if set(df[col].unique()) <= {0, 1, 0.0, 1.0}:
        df[col] = df[col].astype('bool')
        print(f"  → {col}: numeric → bool")

print("\n✓ Datatype Conversion Complete")

print("\nOptimized Data Types:")
print(df.dtypes)

# Memory comparison
memory_before = df_raw.memory_usage(deep=True).sum() / 1024**2
memory_after = df.memory_usage(deep=True).sum() / 1024**2
memory_saved = memory_before - memory_after
memory_saved_pct = (memory_saved / memory_before) * 100

print(f"\nMemory Usage Comparison:")
print(f"  Before: {memory_before:.2f} MB")
print(f"  After: {memory_after:.2f} MB")
print(f"  Saved: {memory_saved:.2f} MB ({memory_saved_pct:.1f}%)")

STEP 3: DATATYPE CONVERSION

Current Data Types:
Week                        object
Geo                         object
Brand                       object
SKU                         object
Sales_Units                float64
Sales_Value                float64
MRP                        float64
Net_Price                  float64
Feature_Flag                 int64
Display_Flag                 int64
TPR_Flag                     int64
Trade_Spend                float64
TV_Impressions             float64
YouTube_Impressions        float64
Facebook_Impressions       float64
Instagram_Impressions      float64
Print_Readership           float64
Radio_Listenership         float64
FB_Banner_Content_Score    float64
IG_Banner_Content_Score    float64
Weighted_Distribution      float64
Numeric_Distribution       float64
TDP                        float64
NOS                        float64
CPI                        float64
GDP_Growth                 float64
Festival_Index             float64
Rainfa